## RLDMUU 2025
#### Backward Induction
jakub.tluczek@unine.ch

Today your task would be to implement the backwards induction algorithm for the following MDP (you can also find it in `src/MDP/MDP.py` on our github):

In [1]:
import numpy as np

## This a discrete MDP with a finite number of states and actions
class DiscreteMDP:
    ## initalise a random MDP with
    ## n_states: the number of states
    ## n_actions: the number of actions
    ## Optional arguments:
    ## P: the state-action-state transition matrix so that P[s,a,s_next] is the probability of s_next given the current state-action pair (s,a)
    ## R: The state-action reward matrix so that R[s,a] is the reward for taking action a in state s.
    def __init__(self, n_states, n_actions, P = None, R = None):
        self.n_states = n_states # the number of states of the MDP
        self.n_actions = n_actions # the number of actions of the MDP
        if (P is None):
            self.P = np.zeros([n_states, n_actions, n_states]) # the transition probability matrix of the MDP so that P[s,a,s'] is the probabiltiy of going to s' from (s,a)
            for s in range(self.n_states):
                for a in range(self.n_actions):
                    self.P[s,a] = np.random.dirichlet(np.ones(n_states)) # generalisation of Beta to multiple outcome
        else:
            self.P = P
        if (R is None):
            self.R = np.zeros([n_states, n_actions]) # the expected reward for each action and state
            # generate uniformly random transitions and 0.1 bernoulli rewards
            for s in range(self.n_states):
                for a in range(self.n_actions):
                    self.R[s,a] = np.round(np.random.uniform(), decimals=1)
        else:
            self.R = R
        
        # check transitions
        for s in range(self.n_states):
            for a in range(self.n_actions):
                #print(s,a, ":", self.P[s,a,:])
                assert(abs(np.sum(self.P[s,a,:])-1) <= 1e-3)
                assert((self.P[s,a,:] <= 1).all())
                assert((self.P[s,a,:] >= 0).all())
                
    # get the probability of next state j given current state s, action a, i.e. P(j|s,a)
    def get_transition_probability(self, state, action, next_state):
        return self.P[state, action, next_state]
    
    # get the vector of probabilities over next states P( . | s,a)
    def get_transition_probabilities(self, state, action):
        return self.P[state, action]
    
    # Get the reward for the current state action.
    # It can also be interpreted as the expected reward for the state and action.
    def get_reward(self, state, action):
        return self.R[state, action]

### Backward induction

As a reminder, in the backward induction algorithm we consider an MDP with finite horizon $T$, and for each step of the algorithm, we compute:

$$ V_t(s) = max_{a \in A} \left[ R(s,a) + \sum_{s' \in S} P(s' | s,a)V_{t+1}(s') \right]$$

where $R(s,a)$ is a reward received by picking action $a$ in state $s$, $P(s'|s,a)$ is the probability of transitioning into next state $s'$, and $V_{t+1}(s')$ is the value of said next state at time $t+1$. We can also say, that for the last timestep (with index $T-1$) the next state value is 0 for every state $V_T(s) = 0$. Consecutively, the action $a$ which maximizes $V_t(s)$ can be described as policy $\pi_t(s)$ at state $s$ and time $t$, 

Your task is to implement this algorithm. Remember to do the inverse iteration, and iterate from $T-1$ to $0$, not the other way around. Your function should return matrix of state values for each $s$ and $t$, as well as resulting policy:

In [2]:
# TODO: Implement backwards induction
def backwards_induction(mdp, T):
    n_states = mdp.n_states
    n_actions = mdp.n_actions

    # Initialize value function and policy arrays
    V = np.zeros((T, n_states))
    pi = np.zeros((T, n_states), dtype=int)

    # Backward induction: iterate from T-1 down to 0
    for t in reversed(range(T)):
        for s in range(n_states):
            action_values = np.zeros(n_actions)
            for a in range(n_actions):
                r = mdp.get_reward(s, a)
                if t < T-1:
                    P_next = mdp.get_transition_probabilities(s, a)
                    expected_next = np.dot(P_next, V[t+1])
                else:
                    expected_next = 0
                action_values[a] = r + expected_next

            best_action = np.argmax(action_values)
            V[t, s] = action_values[best_action]
            pi[t, s] = best_action

    return V, pi

In [3]:
STATES = 5
ACTIONS = 3

T = 15

mdp = DiscreteMDP(STATES, ACTIONS)

V, policy = backwards_induction(mdp, T)

print(V)

print(policy)


[[12.10066315 11.85527781 11.4552979  12.01922887 11.64295716]
 [11.30905923 11.06367434 10.66369497 11.22762252 10.85135666]
 [10.51745582 10.27207184  9.8720936  10.43601408 10.05976031]
 [ 9.72585345  9.48047136  9.08049545  9.6444013   9.26817258]
 [ 8.93425324  8.68887507  8.28890397  8.85277954  8.47660268]
 [ 8.1426575   7.89728742  7.49732627  8.06113917  7.68506976]
 [ 7.35107106  7.10571765  6.70577695  7.26946027  6.89361347]
 [ 6.55950404  6.31418519  5.91428601  6.47770146  6.10231593]
 [ 5.76797794  5.52273254  5.12291584  5.6857765   5.31134525]
 [ 4.9765397   4.73146077  4.33180421  4.89350398  4.52103378]
 [ 4.18528427  3.94063984  3.54130076  4.10050014  3.73198495]
 [ 3.3944195   3.15103879  2.75250659  3.30594264  2.94502047]
 [ 2.60384742  2.36474143  1.96930335  2.50826187  2.16060305]
 [ 1.81382959  1.58476279  1.20602398  1.70456014  1.3751228 ]
 [ 1.          0.8         0.5         0.9         0.6       ]]
[[2 2 0 0 1]
 [2 2 0 0 1]
 [2 2 0 0 1]
 [2 2 0 0 1]
 [